In [1]:
# !nvidia-smi

In [2]:
import os.path as osp
import random

import xml.etree.ElementTree as ET

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.utils.data as data
import torchvision

In [3]:
# 랜덤값 고정
torch.manual_seed(111)
np.random.seed(111)
random.seed(111)

In [4]:
# basepath = './data/VOCdevkit/VOC2012/'
# img_data_template = osp.join(basepath, 'JPEGImages', '%s.jpg') # 규칙 설정
# annopath_template = osp.join(basepath, 'Annotations', '%s.xml') # 규칙 설정

In [5]:
# tr_ids = osp.join(basepath, 'ImageSets/Main/train.txt')
# val_ids = osp.join(basepath, 'ImageSets/Main/val.txt')

In [6]:
# tr_img_list = []
# tr_anno_list = []
# for i in open(tr_ids):
#     tr_img_list.append(img_data_template % i.strip())
#     tr_anno_list.append(annopath_template % i.strip())

In [7]:
# val_img_list = []
# val_anno_list = []
# for i in open(val_ids):
#     val_img_list.append(img_data_template % i.strip())
#     val_anno_list.append(annopath_template % i.strip())

위에서 작업한 것을 함수화

In [8]:
def make_datapath_list(root):
    img_data_template = osp.join(basepath, 'JPEGImages', '%s.jpg') # 규칙 설정
    annopath_template = osp.join(basepath, 'Annotations', '%s.xml')
    tr_ids = osp.join(basepath, 'ImageSets/Main/train.txt')
    val_ids = osp.join(basepath, 'ImageSets/Main/val.txt')
    tr_img_list = []
    tr_anno_list = []
    for i in open(tr_ids):
        tr_img_list.append(img_data_template % i.strip())
        tr_anno_list.append(annopath_template % i.strip())
    val_img_list = []
    val_anno_list = []
    for i in open(val_ids):
        val_img_list.append(img_data_template % i.strip())
        val_anno_list.append(annopath_template % i.strip())
    
    return tr_img_list, tr_anno_list, val_img_list, val_anno_list

In [9]:
basepath = './data/VOCdevkit/VOC2012/'
tr_img_list, tr_anno_list, val_img_list, val_anno_list = make_datapath_list(basepath)

In [10]:
ck_xml_path = tr_anno_list[0]

In [11]:
xml = ET.parse(ck_xml_path).getroot()

In [12]:
l = ['a', 'b']
l.index('a')

0

In [13]:
width = 500
height = 442
class_name_list =['aeroplane', 'bicycle', 'bird', 'boat',
'bottle', 'bus', 'car', 'cat', 'chair',
'cow', 'diningtable', 'dog', 'horse',
'motorbike', 'person', 'pottedplant',
'sheep', 'sofa', 'train', 'tvmonitor']
for i in xml.iter('object'):
    difficult = int(i.find('difficult').text)
    if difficult == 1:
        continue
    bndbox = []
    name = i.find('name').text.lower().strip()
    bbox = i.find('bndbox')
    pts = ['xmin', 'ymin', 'xmax', 'ymax']
    for pt in pts:
        cur_pixel = int(bbox.find(pt).text)-1 # 좌표값은 1부터 시작하므로 연산단위에서 쓰는 0으로 맞추기 위해 계산진행
        if pt == 'xmin' or pt == 'ymin' :
            cur_pixel /= width
        else:
            cur_pixel /= height
        bndbox.append(cur_pixel)
    label_index = class_name_list.index(name)
    bndbox.append(label_index)

위 내용을 클래스로 정리

In [14]:
class Anno_xml2list:
    def __init__(self, classes):
        self.classes = classes
    
    def __call__(self, xml_path, width, height):
        ret = []
        xml = ET.parse(xml_path).getroot()
        class_name_list = self.classes
        for i in xml.iter('object'): # 끝나면 바운딩 박스 하나 종료
            difficult = int(i.find('difficult').text)
            if difficult == 1:
                continue
            bndbox = []
            name = i.find('name').text.lower().strip()
            bbox = i.find('bndbox')
            pts = ['xmin', 'ymin', 'xmax', 'ymax']
            for pt in pts:
                cur_pixel = int(bbox.find(pt).text)-1 # 좌표값은 1부터 시작하므로 연산단위에서 쓰는 0으로 맞추기 위해 계산진행
                if pt == 'xmin' or pt == 'ymin' :
                    cur_pixel /= width
                else:
                    cur_pixel /= height
                bndbox.append(cur_pixel)
            label_index = class_name_list.index(name)
            bndbox.append(label_index)
            ret.append(bndbox)
        return np.array(ret)

In [15]:
from pathlib import Path
f_data = Path(basepath+'ImageSets/Main/')
set_v = set()
for i in f_data.iterdir():
    if '_' in i.name:
        set_v.add(i.name.split('_')[0])
sorted(list(set_v))

['aeroplane',
 'bicycle',
 'bird',
 'boat',
 'bottle',
 'bus',
 'car',
 'cat',
 'chair',
 'cow',
 'diningtable',
 'dog',
 'horse',
 'motorbike',
 'person',
 'pottedplant',
 'sheep',
 'sofa',
 'train',
 'tvmonitor']

In [16]:
voc_classes = ['aeroplane', 'bicycle', 'bird', 'boat',
'bottle', 'bus', 'car', 'cat', 'chair',
'cow', 'diningtable', 'dog', 'horse',
'motorbike', 'person', 'pottedplant',
'sheep', 'sofa', 'train', 'tvmonitor']
transform_anno = Anno_xml2list(voc_classes)
idx = 123
img_path = tr_img_list[idx]
img = cv2.imread(img_path)
h,w,c = img.shape
ann_img_tr = transform_anno(tr_anno_list[idx], w, h)
ann_img_tr

array([[ 0.        ,  0.104     ,  0.416     ,  0.54666667, 19.        ],
       [ 0.544     ,  0.478     ,  1.33066667,  0.99733333,  8.        ],
       [ 0.8       ,  0.246     ,  1.15466667,  0.47466667,  4.        ]])

In [17]:
from utils import data_a
class Make_dataset_Transform:
    def __init__(self, input_size, color_mean):
        self.base_transform = {
            'train': data_a.Compose([
                data_a.ConvertFromInts(), # int -> float
                data_a.ToAbsoluteCoords(), # 어노테이션 데이터의 규격화
                data_a.PhotometricDistort(), # 랜덤한 색조 변화
                data_a.RandomSampleCrop(), # 이미지 랜덤 샘플화(이미지 증강구조 추가 가능)
                data_a.ToPercentCoords(), # 어노테이션 데이터의 0~1크기의 규격화
                data_a.Resize(input_size), # 크기 변경
                data_a.SubtractMeans(color_mean) # 평균값 차연산
            ]),
            'val': data_a.Compose([
                data_a.ConvertFromInts(),
                data_a.Resize(input_size),
                data_a.SubtractMeans(color_mean)
            ])
        }

    def __call__(self, img, boxes, labels, phase):
        return self.base_transform[phase](img, boxes, labels)

In [18]:
idx = 0
img_path = tr_img_list[idx]
img = cv2.imread(img_path)
h,w,c = img.shape # 현재는 BGR 구조로 되어 있음
transform_anno = Anno_xml2list(voc_classes)
anno_list = transform_anno(tr_anno_list[idx], w, h)

In [19]:
color_mean = (104, 117, 123) # BGR 평균. 원래는 계산을 하지만 이미 정립된 수치가 있어서 그대로 차용
input_size = 300
tr = Make_dataset_Transform(input_size, color_mean)

# 모델

In [20]:
import os
import urllib.request

In [21]:
weights_dir='./weights'
if not osp.exists(weights_dir):
    os.mkdir(weights_dir)
url1 = "https://s3.amazonaws.com/amdegroot-models/vgg16_reducedfc.pth"
url2 = "https://s3.amazonaws.com/amdegroot-models/ssd300_mAP_77.43_v2.pth"
t_path1 = osp.join(weights_dir, 'vgg16_reducedfc.pth')
t_path2 = osp.join(weights_dir, 'ssd300_mAP_77.43_v2.pth')
if not osp.exists(t_path1):
    urllib.request.urlretrieve(url1, t_path1)
if not osp.exists(t_path2):
    urllib.request.urlretrieve(url2, t_path2)

In [22]:
from math import sqrt
from itertools import product
import pandas as pd
# from torch.autograd import Function # 과거에 쓰이던 코드임
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init

In [24]:
def make_vgg():
    # 그대로 쓸까? 몇몇 레이어만 쓸까? 일부를 새롭게 만들어서 쓸까?
    layers = []
    in_c = 3
    # vgg 모듈 내 합성곱층 맥스풀링층 채널 수 정의
    cfg = [64,64,'M', 128,128,'M',256,256,'MC',512,512,512,'M',512,512,512]
    for i in cfg:
        if i == 'M':
            layers += [nn.MaxPool2d(kernel_size=2, stride=2)]
        elif i == 'MC':
            layers += [nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)] # ceil모드는 소수점에 의미가 있다고 판단하도록 하는 것임
        else:
            conv2d = nn.Conv2d(in_c, i, kernel_size=3, padding=1)
            layers += [conv2d, nn.ReLU(inplace=True)]
            in_c = i
    pool5 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
    conv6 = nn.Conv2d(512, 1024, kernel_size=3, padding=6, dilation=6) # 팽창 - 수용영역 면적을 넓힘
    conv7 = nn.Conv2d(1024, 1024, kernel_size=1)
    layers += [pool5, conv6, nn.ReLU(inplace=True), conv7, nn.ReLU(inplace=True)]
    return nn.ModuleList(layers)
vgg_m_test = make_vgg()
vgg_m_test

ModuleList(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (13): ReLU(inplace=True)
  (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
  (15): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (16): ReLU(inplace=True)
  (17): Conv2d(512, 512, kernel_siz

In [25]:
def make_extras():
    layers = []
    in_c = 1024
    cfg = [256,512,128,256,128,256,128,256]
    layers += [nn.Conv2d(in_c, cfg[0], kernel_size=(1))]
    layers += [nn.Conv2d(cfg[0], cfg[1], kernel_size=(3), stride=2, padding=1)]
    layers += [nn.Conv2d(cfg[1], cfg[2], kernel_size=(1))]
    layers += [nn.Conv2d(cfg[2], cfg[3], kernel_size=(3), stride=2, padding=1)]
    layers += [nn.Conv2d(cfg[3], cfg[4], kernel_size=(1))]
    layers += [nn.Conv2d(cfg[4], cfg[5], kernel_size=(3))]
    layers += [nn.Conv2d(cfg[5], cfg[6], kernel_size=(1))]
    layers += [nn.Conv2d(cfg[6], cfg[7], kernel_size=(3))]
    return nn.ModuleList(layers)
make_extras()

ModuleList(
  (0): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
  (1): Conv2d(256, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (2): Conv2d(512, 128, kernel_size=(1, 1), stride=(1, 1))
  (3): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (4): Conv2d(256, 128, kernel_size=(1, 1), stride=(1, 1))
  (5): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1))
  (6): Conv2d(256, 128, kernel_size=(1, 1), stride=(1, 1))
  (7): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1))
)

In [26]:
def make_loc_onf(class_n=21, bbox_aspect_num=[4,6,6,6,4,4]):
    loc_layers, conf_layers = [], []
    
    loc_layers += [nn.Conv2d(512, bbox_aspect_num[0]*4, kernel_size=3, padding=1)]
    conf_layers += [nn.Conv2d(512, bbox_aspect_num[0]*class_n, kernel_size=3, padding=1)]

    loc_layers += [nn.Conv2d(1024, bbox_aspect_num[1]*4, kernel_size=3, padding=1)]
    conf_layers += [nn.Conv2d(1024, bbox_aspect_num[1]*class_n, kernel_size=3, padding=1)]

    loc_layers += [nn.Conv2d(512, bbox_aspect_num[2]*4, kernel_size=3, padding=1)]
    conf_layers += [nn.Conv2d(512, bbox_aspect_num[2]*class_n, kernel_size=3, padding=1)]

    loc_layers += [nn.Conv2d(256, bbox_aspect_num[3]*4, kernel_size=3, padding=1)]
    conf_layers += [nn.Conv2d(256, bbox_aspect_num[3]*class_n, kernel_size=3, padding=1)]

    loc_layers += [nn.Conv2d(256, bbox_aspect_num[4]*4, kernel_size=3, padding=1)]
    conf_layers += [nn.Conv2d(256, bbox_aspect_num[4]*class_n, kernel_size=3, padding=1)]
    
    loc_layers += [nn.Conv2d(256, bbox_aspect_num[5]*4, kernel_size=3, padding=1)]
    conf_layers += [nn.Conv2d(256, bbox_aspect_num[5]*class_n, kernel_size=3, padding=1)]
    
    return nn.ModuleList(loc_layers), nn.ModuleList(conf_layers)

make_loc_onf()

(ModuleList(
   (0): Conv2d(512, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (1): Conv2d(1024, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (2): Conv2d(512, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (3): Conv2d(256, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (4-5): 2 x Conv2d(256, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
 ),
 ModuleList(
   (0): Conv2d(512, 84, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (1): Conv2d(1024, 126, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (2): Conv2d(512, 126, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (3): Conv2d(256, 126, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
   (4-5): 2 x Conv2d(256, 84, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
 ))

In [27]:
class L2Norm(nn.Module):
    def __init__(self, input_c=512, scale=20): # 4번째 레이어부터 규제 적용함
        super().__init__()
        self.weight = nn.Parameter(torch.Tensor(input_c))
        self.scale = scale
        self.reset_parameter()
        self.eps = 1e-10

    def reset_parameter(self):
        init.constant_(self.weight, self.scale)

    def forward(self, x):
        norm = x.pow(2), sum(dim=1, keepdim=True).sqrt() + self.eps
        x = torch.div(x, norm)
        weights = self.weight.unsqueeze(0).unsqueeze(2).unsqueeze(3).expand_as(x)
        out = weights * x
        return out

In [30]:
ssd_cfg = {
    'num_classes': 21,  # 배경 클래스를 포함한 총 클래스 수
    'input_size': 300,  # 화상의 입력 크기
    'bbox_aspect_num': [4, 6, 6, 6, 4, 4],  # 출력할 Box 화면비의 종류
    'feature_maps': [38, 19, 10, 5, 3, 1],  # 각 source의 화상 크기
    'steps': [8, 16, 32, 64, 100, 300],  # DBOX의 크기를 정한다
    'min_sizes': [30, 60, 111, 162, 213, 264],  # DBOX의 크기를 정한다
    'max_sizes': [60, 111, 162, 213, 264, 315],  # DBOX의 크기를 정한다
    'aspect_ratios': [[2], [2, 3], [2, 3], [2, 3], [2], [2]],
}
class DBox:
    def __init__(self, cfg:dict):
        super().__init__()
        self.img_size = cfg['input_size']
        self.feature_map = cfg['feature_maps']
        self.num_priors = len(cfg['feature_maps'])
        self.steps = cfg['steps']
        self.min_sizes = cfg['min_sizes']
        self.max_sizes = cfg['max_sizes']
        self.aspect_ratios = cfg['aspect_ratios'] # 종횡비

    def make_dbox_list(self):
        mean = []
        for k, f in enumerate(self.feature_map):
            for i, j in product(range(f), repeat=2):
                f_k = self.img_size/self.steps[k]
                cx = (j+0.5)/f_k # 중심 좌표값 찾기
                cy = (i+0.5)/f_k
                
                s_k = self.min_sizes[k]/self.img_size
                mean += [cx, cy, s_k, s_k]
                
                s_k_prime = sqrt(s_k*(self.min_sizes[k]/self.img_size))
                mean += [cx, cy, s_k_prime, s_k_prime]

                for a in self.aspect_ratios[k]: # dbox의 기본 패턴 적용
                    mean += [cx, cy, s_k*sqrt(a), s_k/sqrt(a)]
                    mean += [cx, cy, s_k/sqrt(a), s_k*sqrt(a)]

        output = torch.Tensor(mean).view(-1, 4)
        output.clamp_(max=1, min=0) # 검출 범위가 이미지의 범위를 넘어서지 않도록 작업
        return output
    
dbox = DBox(ssd_cfg)

In [34]:
pd.DataFrame(dbox.make_dbox_list())

,0,1,2,3
0,0.013333,0.013333,0.100000,0.100000
1,0.013333,0.013333,0.100000,0.100000
2,0.013333,0.013333,0.141421,0.070711
3,0.013333,0.013333,0.070711,0.141421
4,0.040000,0.013333,0.100000,0.100000
...,...,...,...,...
8727,0.833333,0.833333,0.502046,1.000000
8728,0.500000,0.500000,0.880000,0.880000
8729,0.500000,0.500000,0.880000,0.880000
8730,0.500000,0.500000,1.000000,0.622254


In [ ]:
class SSD(nn.Module):
    def __init__(self):
        super().__init__()